## Cell 1 — Imports


In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split

pd.set_option("display.max_colwidth", 300)


## Cell 2 — Load dataset


In [ ]:
df = pd.read_csv("data/data.csv")

print("Original shape:", df.shape)
display(df.head())


## Cell 3 — Remove unnecessary column


In [ ]:
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

print(df.columns)


## Cell 4 — Basic cleaning


In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

df["text"] = df["text"].apply(clean_text)


## Cell 5 — Check missing/empty text


In [ ]:
print("Missing text:", df["text"].isna().sum())

empty_mask = df["text"].str.strip().eq("")

print("Empty text:", empty_mask.sum())

df = df[~empty_mask].copy()

print("Shape after removing empty text:", df.shape)


## Cell 6 — Remove exact duplicate texts


In [ ]:
before = len(df)

df = df.drop_duplicates(
    subset=["text"],
    keep="first"
).reset_index(drop=True)

after = len(df)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)


## Cell 7 — IMPORTANT: conflicting labels


In [ ]:
original = pd.read_csv("data/data.csv")

original["text"] = original["text"].apply(clean_text)

conflicting = (
    original.groupby("text")["label"]
    .nunique()
    .reset_index(name="num_labels")
)

conflicting = conflicting[
    conflicting["num_labels"] > 1
]

print("Texts with conflicting labels:", len(conflicting))

conflict_texts = set(conflicting["text"])

display(
    original[
        original["text"].isin(conflict_texts)
    ][["text", "label"]].head(20)
)


## Cell 8 — Check final label distribution


In [ ]:
print(df["label"].value_counts())
print()
print(df["label"].value_counts(normalize=True) * 100)

import matplotlib.pyplot as plt

df["label"].value_counts().sort_index().plot(
    kind="bar",
    figsize=(6, 4)
)

plt.xlabel("Label")
plt.ylabel("Number of articles")
plt.title("Final Class Distribution")
plt.xticks(rotation=0)
plt.show()


## Cell 9 — Recalculate word count


In [ ]:
df["calculated_word_count"] = (
    df["text"]
    .str.split()
    .str.len()
)

df["wcount_difference"] = (
    df["wcount"] - df["calculated_word_count"]
)

print(
    df["wcount_difference"]
    .describe()
)

mismatch = (
    df["wcount_difference"] != 0
).sum()

print("Word count mismatches:", mismatch)


## Cell 10 — Remove obviously unusable text


In [ ]:
short_texts = df[
    df["calculated_word_count"] <= 5
]

print("Texts with <=5 words:", len(short_texts))

display(
    short_texts[
        ["text", "label", "calculated_word_count"]
    ].head(50)
)


## Cell 11 — Final dataset


In [ ]:
final_df = df[
    ["text", "label"]
].copy()

print(final_df.shape)
display(final_df.head())


## Cell 12 — Train/Validation/Test split


In [ ]:
train_df, temp_df = train_test_split(
    final_df,
    test_size=0.30,
    stratify=final_df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


## Cell 13 — Verify class distribution


In [ ]:
def show_distribution(name, data):
    print(f"\n{name}")
    print("-" * 30)
    print(data["label"].value_counts())
    print()
    print(data["label"].value_counts(normalize=True) * 100)

show_distribution("TRAIN", train_df)
show_distribution("VALIDATION", val_df)
show_distribution("TEST", test_df)


## Cell 14 — Check for leakage between splits


In [ ]:
train_texts = set(train_df["text"])
val_texts = set(val_df["text"])
test_texts = set(test_df["text"])

print("Train ∩ Validation:", len(train_texts & val_texts))
print("Train ∩ Test:", len(train_texts & test_texts))
print("Validation ∩ Test:", len(val_texts & test_texts))


## Cell 15 — Save the splits


In [ ]:
import os

os.makedirs("data/processed", exist_ok=True)

train_df.to_csv(
    "data/processed/train.csv",
    index=False
)

val_df.to_csv(
    "data/processed/validation.csv",
    index=False
)

test_df.to_csv(
    "data/processed/test.csv",
    index=False
)

print("Saved successfully.")


## Cell 16 — Final summary


In [ ]:
print("=" * 50)
print("FINAL DATASET SUMMARY")
print("=" * 50)

print("Total:", len(final_df))
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nLabels:")
print(final_df["label"].value_counts())

print("\nTrain labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())
